In [ ]:
# Outliers: Omitted Variable Bias and Effect of Outliers in Regression Analysis

This notebook explores omitted variable bias and the effect of outliers in regression analysis, following the structure and simulations from the original Quarto document.
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 1. Import Required Libraries
Import numpy, pandas, matplotlib, and scikit-learn's LinearRegression.
</VSCode.Cell>
<VSCode.Cell language="python">
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

np.random.seed(42)
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 2. Simulate Omitted Variable Bias
Simulate data for a linear model with two correlated predictors. Generate $X$, $Z$, and $Y$ according to the model:

$$ Y_i = 1 + 2 X_i + 3 Z_i + e_i $$

where $e_i \sim N(0,1)$ and $Z = 0.8 X + 0.6 u$ with $u \sim N(0,1)$ independent of $X$.
</VSCode.Cell>
<VSCode.Cell language="python">
def simulate_and_compare(n=100):
    a, b, c = 1, 2, 3
    X = np.random.normal(0, 1, n)
    u = np.random.normal(0, 1, n)
    Z = 0.8 * X + 0.6 * u  # X and Z are correlated
    e = np.random.normal(0, 1, n)
    Y = a + b * X + c * Z + e
    return X, Z, Y

# Example simulation
X, Z, Y = simulate_and_compare(n=100)
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 3. Compare Full and Reduced Regression Models
Fit a full regression model ($Y$ on $X$ and $Z$) and a reduced model ($Y$ on $X$ only). Print and compare the estimated coefficients for different sample sizes.
</VSCode.Cell>
<VSCode.Cell language="python">
def fit_models(X, Z, Y):
    # Full model
    XZ = np.column_stack([X, Z])
    model_full = LinearRegression().fit(XZ, Y)
    b_hat_full = model_full.coef_[0]
    c_hat_full = model_full.coef_[1]
    # Reduced model (omit Z)
    model_reduced = LinearRegression().fit(X.reshape(-1, 1), Y)
    b_hat_reduced = model_reduced.coef_[0]
    return b_hat_full, c_hat_full, b_hat_reduced

# Try with n=100
X, Z, Y = simulate_and_compare(n=100)
b_full, c_full, b_reduced = fit_models(X, Z, Y)
print(f"n=100: Full model b_hat = {b_full:.2f}, c_hat = {c_full:.2f}; Reduced model b_hat = {b_reduced:.2f}")

# Try with n=1000
X2, Z2, Y2 = simulate_and_compare(n=1000)
b_full2, c_full2, b_reduced2 = fit_models(X2, Z2, Y2)
print(f"n=1000: Full model b_hat = {b_full2:.2f}, c_hat = {c_full2:.2f}; Reduced model b_hat = {b_reduced2:.2f}")
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 4. Simulate Effect of Outliers on Regression
Generate synthetic linear data, fit an OLS regression, then introduce outliers to the response variable and fit a new regression.
</VSCode.Cell>
<VSCode.Cell language="python">
# Generate synthetic data
n = 30
x = np.random.uniform(0, 5, n)
y = 1 + 3 * x + np.random.normal(0, 1, n)
X_ = x.reshape(-1,1)

# Fit regression without outliers
model_clean = LinearRegression()
model_clean.fit(X_, y)

# Introduce outliers
y_outliers = y.copy()
y_outliers[-3:] = [-20, -25, -30]

# Fit regression with outliers
model_outliers = LinearRegression()
model_outliers.fit(X_, y_outliers)
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 5. Visualize Regression Lines With and Without Outliers
Plot the original data, highlight outliers, and show regression lines for both the clean and outlier-affected models.
</VSCode.Cell>
<VSCode.Cell language="python">
plt.figure(figsize=(8, 5))
plt.scatter(x, y, label="Original Data")
plt.scatter(x[-3:], y_outliers[-3:], color='red', label="Outliers", zorder=5)
plt.plot(x, model_clean.predict(X_), label="Fit (No Outliers)", color='green')
plt.plot(x, model_outliers.predict(X_), label="Fit (With Outliers)", color='red', linestyle='--')
plt.legend()
plt.xlabel("x")
plt.ylabel("y")
plt.title("Effect of Outliers on Regression Line")
plt.show()
</VSCode.Cell>
<VSCode.Cell language="markdown">
## 6. Print and Compare Regression Coefficients
Print the slope and intercept for both models to quantify the effect of outliers.
</VSCode.Cell>
<VSCode.Cell language="python">
print(f"No Outliers: Slope = {model_clean.coef_[0]:.2f}, Intercept = {model_clean.intercept_:.2f}")
print(f"With Outliers: Slope = {model_outliers.coef_[0]:.2f}, Intercept = {model_outliers.intercept_:.2f}")
